# Dependências e versões

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="00_validacao_ambiente"


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': None,
 'model_name': 'planilha-turmas-demo',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 3,
 'campaign_display_name': 'POC DEMO • Validação manual',
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. SQL e tracking real com pequeno artefato diagnóstico. Esperado: tabela PASS, antes de qualquer treino.

In [2]:
from uuid import uuid4
from src.workbench.journey_steps import dependencies, require_dependencies
checks=dependencies(cfg,str(uuid4()))
display(pd.DataFrame(checks)); require_dependencies(checks,cfg)
evidence["dependencies"]=checks


,dependency,status,attempts
0,source SQL,PASS,1
1,application SQL,PASS,1
2,MLflow PostgreSQL + proxy artifact roundtrip,PASS,1


2. Serviços, storage e versões. API /ready valida RabbitMQ; falha resulta NOT_READY.

In [3]:
from src.workbench.exploration import service_checks
import mlflow,sklearn,numpy
services=service_checks();display(pd.DataFrame(services))
assert all(s["status"]=="PASS" for s in services), "NOT_READY: veja diagnóstico por serviço"
evidence.update(services=services,versions={"mlflow":mlflow.__version__,"sklearn":sklearn.__version__,"numpy":numpy.__version__})


,service,status,diagnosis,evidence
0,API,PASS,docker compose logs --tail 80 API,NaN
1,inference,PASS,docker compose logs --tail 80 inference,NaN
2,intake,PASS,docker compose logs --tail 80 intake,NaN
3,dashboard,PASS,docker compose logs --tail 80 dashboard,NaN
4,RabbitMQ,PASS,NaN,API /ready performs broker ensure_connection
5,RustFS,PASS,NaN,S3 head_bucket


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [4]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v3/00_validacao_ambiente.json'